In [ ]:
"""
Step 1: Hardware Connection and Manual Control
===============================================
Features:
- Connect to remote hardware service (SLM + motorized stage + rotation stage)
- Check current position status
- Upload test phase pattern to verify SLM connection
- Set rotation stage to correct angle

Note: Ensure RPyC hardware service is running (port 18861) before execution
"""

from hardware import RemoteHardwareManager
import numpy as np

# ============================================================
# Connect to Remote Hardware Service
# ============================================================
# RemoteHardwareManager manages all hardware:
#   - SLM control
#   - Z-axis translation stage (Z825B)
#   - Rotation stage (PRM1-Z8)
#   - AutoHotkey mouse control (for camera triggering)
hw = RemoteHardwareManager(host="127.0.0.1", port=18861)

# ============================================================
# Check Current Position
# ============================================================
# Get Z-axis position (unit: mm)
initial_z_pos = hw.stage_get_position()
print(f"Current Z-axis position: {initial_z_pos:.4f} mm")

# Get rotation stage angle (unit: degrees)
angle = hw.rotation_get_position()
print(f"Current rotation angle: {angle:.2f} deg")

# ============================================================
# Test SLM Connection: Upload Fresnel Test Pattern
# ============================================================
from phase_generators import PhaseGenerator
from optics_utils import load_dict_from_json

# Load config and generate test phase pattern
params = load_dict_from_json(r".\\config\\base.json")
params['M'] = 5  # Set to 5x5 microlens array

Optimizer = PhaseGenerator(params)
Optimizer.generate(mode='fresnel')  # Generate Fresnel lens phase
phase_8bit = Optimizer.update_phase_8bit()

# Upload to SLM
hw.upload_slm(phase_8bit)
print(f"Test pattern (M={params.get('M', 'default')}) uploaded to SLM")

# ============================================================
# Common Commands Reference (uncomment to use)
# ============================================================

# --- Z-axis Translation Stage Control ---
# Home:
# hw.stage_home()

# Move to specified position (mm):
# hw.stage_move_to(11.805)
# pos = hw.stage_get_position()
# print(f"Position: {pos:.4f} mm")

# ============================================================
# Rotation Stage Angle Presets
# ============================================================
# Standard angles for different M values (based on optical path calibration)
M_ANGLE = {
    'M3': 282.75,  # Angle for 3x3 array
    'M5': 268.0,   # Angle for 5x5 array
    'M7': 261.6,   # Angle for 7x7 array
    'M9': 258.5    # Angle for 9x9 array
}

# Set rotation stage to target angle
target_angle = M_ANGLE['M'+str(params['M'])]
# Overshoot then return to eliminate gear backlash
hw.rotation_move_to(target_angle + 10)
hw.rotation_move_to(target_angle)
print(f"Rotation stage moved to: {hw.rotation_get_position():.2f} deg")

## Test Different Fresnel Configurations

In [ ]:
from phase_generators import PhaseGenerator
from optics_utils import load_dict_from_json
from hardware import RemoteHardwareManager

# Load config and generate test phase pattern
params = load_dict_from_json(r".\\config\\base.json")
params['M'] = 9  # Set microlens array size: 5x5 / 7x7 / 9x9

# Generate Fresnel lens phase and upload to SLM
Optimizer = PhaseGenerator(params)
Optimizer.generate(mode='fresnel') 
phase_8bit = Optimizer.update_phase_8bit()
hw.upload_slm(phase_8bit)
print(f"Test pattern (M={params.get('M', 'default')}) uploaded to SLM")

# Set rotation stage to target angle
label = 'M' + str(params['M'])  # e.g., 'M5'
target_angle = M_ANGLE[label]
# Overshoot then return to eliminate gear backlash
hw.rotation_move_to(target_angle + 10)
hw.rotation_move_to(target_angle)
print(f"Rotation stage moved to: {hw.rotation_get_position():.2f} deg")

# PSF Automated Capture Workflow

## Overview
This notebook automates Point Spread Function (PSF) data acquisition with SLM phase patterns and Z-axis scanning imaging.

## Workflow
1. **Hardware Connection** - Connect SLM, motorized stage, rotation stage; test communication
2. **Select Phase Patterns** - Choose `.npy` phase pattern files from `./output/`
3. **Z-Scan Capture** - Automatically execute Z-axis scan and trigger camera acquisition
4. **Organize Data** - Sort TIFF files by phase pattern name and generate metadata

## Hardware Requirements
- SLM (Meadowlark)
- Z-axis motorized stage (Thorlabs Z825B)
- Rotation stage (Thorlabs PRM1-Z8)
- Camera software (triggered via AutoHotkey)
- RPyC hardware service (port 18861)

## Data Save Location
Default save to NAS: `Z:\SLM_super_resolution\data\for_auto_scan\`

---

In [ ]:
"""
Step 2: Select Phase Pattern Files
===================================
Features:
- Scan ./output/ directory for .npy phase pattern files
- Provide multi-select GUI interface
- Automatically parse M value from filename (e.g., M5, M7)

Usage:
- Ctrl + Click for multiple selection
- Filenames must contain M%d pattern to be recognized (e.g., opt1_M5_xxx.npy)
"""

from npy_file_selector import select_npy_files

# ============================================================
# Create File Selector GUI
# ============================================================
# select_npy_files parameters:
#   output_dir: Directory to search for .npy files

file_selector_widget = select_npy_files(output_dir="./output")

# ============================================================
# After selection, retrieve results using these methods:
# ============================================================
# Get list of selected filenames:
#   selected_files = file_selector_widget.get_selected_files()
#
# Get corresponding M values (e.g., ['M5', 'M5', 'M7']):
#   m_patterns = file_selector_widget.get_m_patterns()
#
# Get output directory path:
#   output_dir = file_selector_widget.get_output_dir()

In [ ]:
"""
Step 3: Z-Scan Automated Capture
=================================
Features:
- Automatically identify phase pattern M value (M3/M5/M7/M9)
- Auto-select scan parameters (num_steps, z_range) based on M value
- Auto-set rotation stage angle and upload SLM phase pattern before scan
- Trigger camera acquisition via AutoHotkey
- Record complete scan parameters for subsequent analysis

Notes:
- Ensure camera software is open and ready for triggering before running
- First run will prompt to click camera trigger button to capture position
- Do not operate mouse during scanning
"""

from hardware import RemoteHardwareManager
import numpy as np
import time
import os
from datetime import datetime
from nas_mapper import quick_map

# ============================================================
# NAS Network Drive Mapping (if saving to NAS)
# ============================================================
success, msg = quick_map()
if not success:
    raise RuntimeError(f"NAS mapping failed: {msg}")

# ============================================================
# Scan Parameter Configuration *** Modify as needed ***
# ============================================================

# Data save directory (Z: drive = NAS path)
# save_dir = r"Z:\\SLM_super_resolution\\data\\for_auto_scan\\"
save_dir = r"Z:\\SLM_super_resolution\\data\\260106_usaf_24_recipes\\"


# Focal plane position (mm) - Center position for Z scan
# z_focal_plane = 11.805
z_focal_plane = initial_z_pos  # Use current position from Step 1

# ----- Scan parameters for different M values -----
# Format: 'M{n}': {'num_steps': steps, 'z_range': scan range (mm)}
# Note: Larger M values need larger scan range to cover greater depth of focus

# # For PSF capture
# SCAN_PARAMS = {
#     'M3': {'num_steps': 81, 'z_range': 0.3},   # 3x3 array
#     'M5': {'num_steps': 81, 'z_range': 0.4},   # 5x5 array
#     'M7': {'num_steps': 81, 'z_range': 0.6},   # 7x7 array
#     'M9': {'num_steps': 81, 'z_range': 0.7},   # 9x9 array
# }

# For sample imaging
SCAN_PARAMS = {
    'M3': {'num_steps': 41, 'z_range': 0.3},   # 3x3 array
    'M5': {'num_steps': 41, 'z_range': 0.4},   # 5x5 array
    'M7': {'num_steps': 41, 'z_range': 0.6},   # 7x7 array
    'M9': {'num_steps': 41, 'z_range': 0.7},   # 9x9 array
}


# ============================================================
# Get Selected File Info from Step 2
# ============================================================
# get_selected_files() returns filename list (for display and logging)
# get_selected_paths() returns full path list (for actual file loading)
# get_m_patterns() returns M value list
selected_npy_files = file_selector_widget.get_selected_files()  # Filename list
selected_npy_paths = file_selector_widget.get_selected_paths()  # Full path list
m_list = file_selector_widget.get_m_patterns()                  # M value list (e.g., ['M5', 'M5', 'M7'])

# ============================================================
# Validate File Selection
# ============================================================
if not selected_npy_files:
    raise ValueError("No .npy files selected! Please return to Step 2 to select files.")

# Verify all M values have corresponding scan parameters
for m in set(m_list):
    if m not in SCAN_PARAMS:
        raise ValueError(f"Unknown M value: {m}, please add configuration in SCAN_PARAMS")
    if m not in M_ANGLE:
        raise ValueError(f"Unknown M value: {m}, please add configuration in M_ANGLE")

print(f"Selected {len(selected_npy_files)} phase pattern files:")
print(f"{'─'*60}")
for i, (name, path, m) in enumerate(zip(selected_npy_files, selected_npy_paths, m_list)):
    params = SCAN_PARAMS[m]
    print(f"   [{i+1}] {name}")
    print(f"       Path: {path}")
    print(f"       -> {m}: {params['num_steps']} steps, +/-{params['z_range']/2:.2f} mm, angle {M_ANGLE[m]} deg")

# ============================================================
# Calculate Total Frames (different M may have different step counts)
# ============================================================
total_frames = sum(SCAN_PARAMS[m]['num_steps'] for m in m_list)
print(f"\nScan Statistics:")
print(f"   Total phase patterns: {len(selected_npy_files)}")
print(f"   Total frames: {total_frames}")

# Statistics by M value
m_counts = {}
for m in m_list:
    m_counts[m] = m_counts.get(m, 0) + 1
for m, count in sorted(m_counts.items()):
    params = SCAN_PARAMS[m]
    print(f"   {m}: {count} phase patterns x {params['num_steps']} steps = {count * params['num_steps']} frames")

# ============================================================
# Connect Hardware
# ============================================================
hw = RemoteHardwareManager(host="127.0.0.1", port=18861)

# ============================================================
# Capture Camera Trigger Button Position
# ============================================================
print("\nCapturing camera trigger button position...")
print("   Please click the [Capture/Acquire] button in camera software...")
click_pos = hw.capture_position()
if click_pos is None:
    raise RuntimeError("Failed to capture click position!")

# ============================================================
# Initialize Scan Record
# ============================================================
scan_start_time = datetime.now()
scan_info = {
    'start_time': scan_start_time.isoformat(),
    'z_focal_plane': z_focal_plane,
    'scan_params_by_m': SCAN_PARAMS,
    'm_angles': M_ANGLE,
    'patterns': [],  # Detailed info for each pattern
    'save_dir': save_dir,
}

# ============================================================
# Clean Old TIFF Files in Temp Folder
# ============================================================
old_files = [f for f in os.listdir(save_dir) if f.endswith((".tiff", ".tif"))]
for file in old_files:
    os.remove(os.path.join(save_dir, file))
print(f"Cleaned {len(old_files)} old TIFF files")

# ============================================================
# Main Scan Loop
# ============================================================
frame_counter = 0
current_m = None  # Track current M value to determine if rotation stage switch needed

print(f"\n{'='*60}")
print(f"Starting Scan")
print(f"{'='*60}")

for pattern_idx, (npy_name, npy_path) in enumerate(zip(selected_npy_files, selected_npy_paths)):
    # --- Get current phase pattern's M value and parameters ---
    m_pattern = m_list[pattern_idx]
    params = SCAN_PARAMS[m_pattern]
    num_steps = params['num_steps']
    z_range = params['z_range']
    target_angle = M_ANGLE[m_pattern]
    
    # Calculate Z position sequence for this pattern
    z_positions = np.linspace(
        z_focal_plane - z_range / 2,
        z_focal_plane + z_range / 2,
        num_steps
    )
    z_step = z_positions[1] - z_positions[0] if len(z_positions) > 1 else 0
    
    print(f"\n[Pattern {pattern_idx+1}/{len(selected_npy_files)}] {npy_name}")
    print(f"   {m_pattern}: {num_steps} steps, range +/-{z_range/2:.3f} mm, step size {z_step*1000:.2f} um")
    
    # ============================================================
    # Pre-scan Setup: Rotation Stage Angle (switch only when M changes)
    # ============================================================
    if m_pattern != current_m:
        print(f"   Switching rotation stage: {current_m} -> {m_pattern} (target angle: {target_angle} deg)")
        hw.rotation_move_to(target_angle + 10)  # Overshoot to eliminate gear backlash
        hw.rotation_move_to(target_angle)
        actual_angle = hw.rotation_get_position()
        print(f"   Rotation stage ready: {actual_angle:.2f} deg")
        current_m = m_pattern
    
    # ============================================================
    # Pre-scan Setup: Upload Phase Pattern to SLM
    # ============================================================
    # Use full path to load file
    pattern = np.load(npy_path)
    hw.upload_slm(pattern)
    print(f"   Phase pattern uploaded to SLM (from {npy_path})")
    
    # ============================================================
    # Record Scan Info for This Pattern
    # ============================================================
    pattern_info = {
        'name': npy_name,
        'path': npy_path,
        'm_pattern': m_pattern,
        'num_steps': num_steps,
        'z_range': z_range,
        'z_positions': z_positions.tolist(),
        'z_step': z_step,
        'rotation_angle': target_angle,
        'frame_start': frame_counter + 1,  # Starting frame number for this pattern
    }
    
    # --- Initialize Z-axis position ---
    hw.stage_move_to(z_positions[0])
    time.sleep(0.2)

    # ============================================================
    # Z-axis Scan Sub-loop
    # ============================================================
    for z_idx, z_pos in enumerate(z_positions):
        frame_counter += 1
        
        # Move Z-axis
        hw.stage_move_to(z_pos)
        time.sleep(0.2)  # Wait for motor to stabilize
        
        # Trigger camera acquisition
        hw.click_at()
        time.sleep(0.65)  # Wait for acquisition to complete
        
        # Display progress
        progress = frame_counter / total_frames * 100
        print(f"\r   Z[{z_idx+1}/{num_steps}] = {z_pos:.4f} mm | "
              f"Frame {frame_counter}/{total_frames} ({progress:.1f}%)", end='')
    
    print()  # Newline
    
    # Record ending frame number for this pattern
    pattern_info['frame_end'] = frame_counter
    scan_info['patterns'].append(pattern_info)

# ============================================================
# Scan Complete, Record End Time
# ============================================================
scan_end_time = datetime.now()
scan_info['end_time'] = scan_end_time.isoformat()
scan_info['duration_seconds'] = (scan_end_time - scan_start_time).total_seconds()
scan_info['total_frames'] = frame_counter

print(f"\n{'='*60}")
print(f"Scan Complete!")
print(f"   Completion time: {scan_end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Total duration: {scan_info['duration_seconds']:.1f} seconds")
print(f"   Frames captured: {frame_counter}")
print(f"\nScan Summary by M value:")
for m in sorted(set(m_list)):
    params = SCAN_PARAMS[m]
    count = m_list.count(m)
    print(f"   {m}: {count} phase patterns, steps={params['num_steps']}, range=+/-{params['z_range']/2:.2f}mm")
print(f"{'='*60}")

In [ ]:
"""
Step 4: Organize Captured Data
===============================
Features:
- Organize TIFF files based on pattern info recorded in scan_info
- Support different frame counts for different M values
- Filename includes M value, Z position, and other complete info
- Generate JSON format scan metadata files

Output Structure:
  save_dir/
  ├── {pattern_name}/
  │   ├── {prefix}_{M}_frame001_z11.6050mm.tiff
  │   ├── {prefix}_{M}_frame002_z11.6100mm.tiff
  │   ├── ...
  │   └── {pattern_name}_scan_info.json
  ├── {pattern_name_2}/
  │   └── ...
  └── scan_log_{timestamp}.json   # Master scan log
"""

import os
import glob
import shutil
import json
from datetime import datetime

# ============================================================
# User Parameter Settings
# ============================================================
# Filename prefix (describes sample or experimental conditions)
# user_prefix = "PSF_4um"
user_prefix = "USAF_sample"    

# ============================================================
# Use Scan Info from Step 3
# ============================================================
# Required from previous cell: scan_info, save_dir



print(f"Organizing data directory: {save_dir}")
print(f"   Filename prefix: {user_prefix}")

# ============================================================
# Find Captured TIFF Files
# ============================================================
tiff_pattern = os.path.join(save_dir, "ss_single_*.tiff")
tiff_files = sorted(
    glob.glob(tiff_pattern), 
    key=lambda x: int(os.path.basename(x).replace('ss_single_', '').replace('.tiff', ''))
)

expected_frames = scan_info['total_frames']
print(f"   Found {len(tiff_files)} TIFF files (expected: {expected_frames})")

if len(tiff_files) != expected_frames:
    print(f"Warning: File count mismatch!")

# ============================================================
# Organize Files by Phase Pattern (using detailed info from scan_info)
# ============================================================
frame_idx = 0

for pattern_idx, pattern_info in enumerate(scan_info['patterns']):
    npy_name = pattern_info['name']
    m_pattern = pattern_info['m_pattern']
    num_steps = pattern_info['num_steps']
    z_positions = pattern_info['z_positions']
    z_range = pattern_info['z_range']
    z_step = pattern_info['z_step']
    
    # Create subfolder named after the phase pattern
    pattern_basename = os.path.splitext(npy_name)[0]
    pattern_folder = os.path.join(save_dir, pattern_basename)
    os.makedirs(pattern_folder, exist_ok=True)
    
    print(f"\n[{pattern_idx+1}/{len(scan_info['patterns'])}] {pattern_basename}/")
    print(f"   {m_pattern}: {num_steps} frames, range +/-{z_range/2:.3f} mm")
    
    # --- Move and rename TIFF files ---
    moved_count = 0
    for z_idx, z_pos in enumerate(z_positions):
        if frame_idx >= len(tiff_files):
            print(f"   Warning: Missing frame {frame_idx+1}")
            frame_idx += 1
            continue
        
        src_path = tiff_files[frame_idx]
        
        # New filename format: {prefix}_{M}_frame{n}_z{position}mm.tiff
        # Includes M value for easier downstream processing
        new_name = f"{user_prefix}_{m_pattern}_frame{z_idx+1:03d}_z{z_pos:.4f}mm.tiff"
        dst_path = os.path.join(pattern_folder, new_name)
        
        shutil.move(src_path, dst_path)
        frame_idx += 1
        moved_count += 1
    
    print(f"   Moved {moved_count} frames")
    
    # --- Generate scan info file for this phase pattern ---
    info_filename = f"{pattern_basename}_scan_info.json"
    info_path = os.path.join(pattern_folder, info_filename)
    
    pattern_scan_info = {
        'pattern_name': npy_name,
        'm_pattern': m_pattern,
        'user_prefix': user_prefix,
        'scan_start_time': scan_info['start_time'],
        'scan_end_time': scan_info['end_time'],
        'duration_seconds': scan_info['duration_seconds'],
        'z_focal_plane_mm': scan_info['z_focal_plane'],
        'z_range_mm': z_range,
        'num_steps': num_steps,
        'z_positions_mm': z_positions,
        'z_step_mm': z_step,
        'rotation_angle_deg': pattern_info['rotation_angle'],
        'frame_range': [pattern_info['frame_start'], pattern_info['frame_end']],
        'file_naming': f"{user_prefix}_{m_pattern}_frame{{n:03d}}_z{{z:.4f}}mm.tiff",
        'total_frames': num_steps,
    }
    
    with open(info_path, 'w', encoding='utf-8') as f:
        json.dump(pattern_scan_info, f, indent=2, ensure_ascii=False)
    
    print(f"   Saved {info_filename}")

# ============================================================
# Generate Master Scan Log
# ============================================================
master_log_path = os.path.join(save_dir, f"scan_log_{scan_start_time.strftime('%Y%m%d_%H%M%S')}.json")

master_info = {
    'scan_start_time': scan_info['start_time'],
    'scan_end_time': scan_info['end_time'],
    'duration_seconds': scan_info['duration_seconds'],
    'user_prefix': user_prefix,
    'z_focal_plane_mm': scan_info['z_focal_plane'],
    'scan_params_by_m': scan_info['scan_params_by_m'],
    'm_angles': scan_info['m_angles'],
    'total_patterns': len(scan_info['patterns']),
    'total_frames': scan_info['total_frames'],
    'patterns': [
        {
            'name': p['name'],
            'm_pattern': p['m_pattern'],
            'num_steps': p['num_steps'],
            'z_range': p['z_range'],
            'folder': os.path.splitext(p['name'])[0],
        }
        for p in scan_info['patterns']
    ],
}

with open(master_log_path, 'w', encoding='utf-8') as f:
    json.dump(master_info, f, indent=2, ensure_ascii=False)

# ============================================================
# Print Summary
# ============================================================
print(f"\n{'='*60}")
print(f"Data Organization Complete!")
print(f"   Master scan log: {os.path.basename(master_log_path)}")
print(f"   Folders created: {len(scan_info['patterns'])}")
print(f"\nSummary by M value:")
m_summary = {}
for p in scan_info['patterns']:
    m = p['m_pattern']
    if m not in m_summary:
        m_summary[m] = {'count': 0, 'frames': 0}
    m_summary[m]['count'] += 1
    m_summary[m]['frames'] += p['num_steps']

for m in sorted(m_summary.keys()):
    s = m_summary[m]
    print(f"   {m}: {s['count']} phase patterns, {s['frames']} frames total")

print(f"\nData location: {save_dir}")
print(f"{'='*60}")